# 3. 컨텍스트 엔지니어링

컨텍스트는 매 턴 비용이 드는 제한된 자원이다.
증가량을 측정하고 압축, 오프로딩, 검색으로 줄인다.

### 공통 설정

`1`과 같은 블록이다. 요약용 하위 모델과 임베딩 모델을 더 쓴다.

In [1]:
import json
import os
import shutil
from pathlib import Path
import numpy as np
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]
EMBEDDING = os.environ["OPENAI_EMBEDDING_MODEL"]
client = OpenAI()

PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50), EMBEDDING: (0.02, 0.0)}
USAGE = []

def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

세 문서를 읽고 정리하는 과제다. `read_doc`은 본문 전체를 돌려준다.

In [3]:
CORPUS = Path("../../../../../data/corpus/graph")
TARGETS = ["한인애국단", "윤봉길", "이봉창_의거"]
TASK = "한인애국단, 윤봉길, 이봉창_의거 세 문서를 읽고 김구가 각각에서 한 일을 한 문장씩 정리한다"


def read_doc(title):
    f = CORPUS / f"{title}.md"
    if not f.exists():
        raise FileNotFoundError(f"'{title}' 문서가 없다. 이 과제의 문서는 {TARGETS} 이다")
    return f.read_text(encoding="utf-8")


print("문서 크기", {t: f"{len(read_doc(t)):,}자" for t in TARGETS})

문서 크기 {'한인애국단': '6,032자', '윤봉길': '7,774자', '이봉창_의거': '5,044자'}


도구 정의는 `2`의 `spec`으로 만든다.

In [4]:
def spec(name, desc, props):
    return {"type": "function", "name": name, "description": desc, "strict": True,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}


TITLE = {"title": {"type": "string", "description": "문서 제목"}}
READ = [spec("read_doc", "문서 한 편의 본문 전체를 돌려준다.", TITLE)]

### 컨텍스트를 직접 관리한다

`store=False`와 입력 배열을 사용하면 컨텍스트를 확인하고 수정할 수 있다. 이 배열이 `5`의 Session 인터페이스가 된다.
턴이 끝날 때마다 `on_turn`이 배열을 고칠 기회를 갖는다.

In [36]:
def run(task, tools, registry, max_turns=8, on_turn=None):
    messages = [{"role": "user", "content": task}]
    trace = []
    for turn in range(1, max_turns + 1):
        r = keep(client.responses.create(model=MODEL, input=messages, tools=tools, store=False, parallel_tool_calls=True))    # store=False : previous id를 쓰지 않음
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "입력토큰": r.usage.input_tokens, "도구": [c.name for c in calls]})
        if not calls:
            return r.output_text, trace
        messages += [i.model_dump() for i in r.output]
        for c in calls:
            try:
                body = registry(c.name, json.loads(c.arguments))
            except Exception as e:
                body = f"error: {e}"
            messages.append({"type": "function_call_output", "call_id": c.call_id, "output": body})
        if on_turn:
            messages = on_turn(messages)    # on_turn 메소드를 통해 context를 가공해서 붙여줌 (압축, 정리 등)
    return None, trace

트레이스에서 턴별 입력 토큰과 합계를 읽는 함수다. 합계를 돌려주므로 뒤에서 방법끼리 비교한다.

In [25]:
def show(label, answer, trace):
    for t in trace:
        print(f"  턴 {t['턴']}  입력 {t['입력토큰']:7,d} 토큰  {t['도구']}")
    total = sum(t["입력토큰"] for t in trace)
    print(f"{label:6} 턴 {len(trace)}  입력 합계 {total:7,d} 토큰   {str(answer)}")
    return total

### 컨텍스트는 도구 결과로 늘어난다.

도구가 돌려준 본문이 다음 턴의 입력에 그대로 사용된다.
읽은 것을 버리지 않으므로 턴이 갈수록 같은 내용을 반복해서 다시 보낸다.

In [26]:
def plain(name, args):
    return read_doc(**args)


answer, trace = run(TASK, READ, plain)
print(answer)

- **한인애국단 문서**: 김구는 1931년 상하이에서 한인애국단을 결성하고 단장으로서 이봉창·윤봉길 등의 의거를 기획·지휘하였다.  
- **윤봉길 문서**: 김구는 윤봉길을 한인애국단에 가입시킨 뒤 홍커우 공원 의거를 계획하고 실행을 맡겼다.  
- **이봉창 의거 문서**: 김구는 이봉창을 모집해 한인애국단에 가입시키고 일본 천황 암살을 위한 의거를 준비·후원하였다.


턴마다 입력 토큰이 어떻게 늘었는지 본다.

In [27]:
base = show("그대로", answer, trace)

  턴 1  입력     167 토큰  ['read_doc']
  턴 2  입력   4,321 토큰  ['read_doc']
  턴 3  입력   9,585 토큰  ['read_doc']
  턴 4  입력   9,656 토큰  ['read_doc']
  턴 5  입력  12,725 토큰  []
그대로    턴 5  입력 합계  36,454 토큰   - **한인애국단 문서**: 김구는 1931년 상하이에서 한인애국단을 결성하고 단장으로서 이봉창·윤봉길 등의 의거를 기획·지휘하였다.  
- **윤봉길 문서**: 김구는 윤봉길을 한인애국단에 가입시킨 뒤 홍커우 공원 의거를 계획하고 실행을 맡겼다.  
- **이봉창 의거 문서**: 김구는 이봉창을 모집해 한인애국단에 가입시키고 일본 천황 암살을 위한 의거를 준비·후원하였다.


### 1. 압축. 오래된 결과를 요약으로 바꾼다

지난 턴의 도구 결과는 대개 다시 읽히지 않는다. 그런데 매 턴 비용을 지불한다.
마지막 둘만 원문으로 두고 나머지는 요약으로 갈아 끼운다. 요약은 하위 모델로 한다.

In [28]:
def compact(messages, keep_last=2):
    spots = [i for i, m in enumerate(messages) if m.get("type") == "function_call_output"]
    for i in spots[:-keep_last]:
        body = messages[i]["output"]
        if len(body) < 1500 or body.startswith("[압축됨]"): # body의 길이에 따라 길어지면 llm 통해 압축
            continue
        s = keep(client.responses.create(
            model=LOWER, store=False,
            input=f"다음 글에서 김구와 관련된 사실만 세 문장으로 추린다.\n\n{body[:8000]}")).output_text
        messages[i] = {**messages[i], "output": "[압축됨] " + s}
    return messages

`on_turn`에 걸고 같은 과제를 돌린다.

In [30]:
answer, trace = run(TASK, READ, plain, on_turn=compact)
print(answer)

- **한인애국단**: 김구는 1931년 상하이에서 대한민국 임시정부 주석으로서 한인애국단을 조직했다.  
- **윤봉길**: 김구는 윤봉길을 한인애국단에 가입시킨 뒤 홍커우 공원 의거를 윤봉길과 협의해 계획했다.  
- **이봉창_의거**: 김구는 이봉창의 천황 공격 제안을 받아들여 그를 한인애국단에 모집하고 폭탄과 자금을 마련해 거사를 준비시켰다.


In [31]:
packed = show("압축", answer, trace)

  턴 1  입력     167 토큰  ['read_doc']
  턴 2  입력   4,321 토큰  ['read_doc']
  턴 3  입력   9,585 토큰  ['read_doc']
  턴 4  입력   5,688 토큰  ['read_doc']
  턴 5  입력   3,680 토큰  []
압축     턴 5  입력 합계  23,441 토큰   - **한인애국단**: 김구는 1931년 상하이에서 대한민국 임시정부 주석으로서 한인애국단을 조직했다.  
- **윤봉길**: 김구는 윤봉길을 한인애국단에 가입시킨 뒤 홍커우 공원 의거를 윤봉길과 협의해 계획했다.  
- **이봉창_의거**: 김구는 이봉창의 천황 공격 제안을 받아들여 그를 한인애국단에 모집하고 폭탄과 자금을 마련해 거사를 준비시켰다.


### 2. 오프로딩. 큰 결과는 파일에 두고 경로만 남긴다

압축은 되돌릴 수 없다. 줄인 만큼 잃는다.
파일에 쓰면 컨텍스트에서는 빠지지만 없어지지는 않는다. 필요하면 구간을 다시 가져온다.
코딩 에이전트가 파일시스템을 상태로 쓰는 것이 이 방식이다(Part 5 `21`).

In [32]:
SPOOL = Path("spool")
SPOOL.mkdir(exist_ok=True)


def read_offload(title):
    text = read_doc(title)
    (SPOOL / f"{title}.txt").write_text(text, encoding="utf-8")
    return (f"spool/{title}.txt 에 {len(text):,}자를 저장했다."
            f" 더 필요하면 read_span 으로 구간을 가져온다.\n앞 500자:\n{text[:500]}")


def read_span(path, start, length):
    text = Path(path).read_text(encoding="utf-8")
    return text[start:start + length]


print(read_offload("윤봉길")[:120])

spool/윤봉길.txt 에 7,774자를 저장했다. 더 필요하면 read_span 으로 구간을 가져온다.
앞 500자:
# 윤봉길

윤봉길(尹奉吉, 1908년 6월 21일~1932년 12월 19일)은 대한제국의 독


`read_doc`의 설명이 바뀌었다. 파일 위치와 앞부분만 준다는 것을 모델이 알아야 `read_span`을 부른다.

In [33]:
OFFLOAD = [spec("read_doc", "문서를 파일로 내보내고 저장 위치와 앞부분을 돌려준다.", TITLE),
           spec("read_span", "파일의 특정 구간을 돌려준다.",
                {"path": {"type": "string", "description": "spool/ 아래 경로"},
                 "start": {"type": "integer", "description": "시작 위치"},
                 "length": {"type": "integer", "description": "가져올 글자 수"}})]


def offload(name, args):
    return read_offload(**args) if name == "read_doc" else read_span(**args)

In [37]:
answer, trace = run(TASK, OFFLOAD, offload)
print(answer)

- **한인애국단**: 김구는 1931년 상하이에서 대한민국 임시정부 내 항일무장투쟁단체인 한인애국단을 조직하고 단장으로서 단원 모집과 거사를 지휘했다.  
- **윤봉길**: 김구는 윤봉길을 한인애국단에 가입시킨 뒤 1932년 상하이 홍커우 공원 의거를 계획하고 실행하도록 맡겼다.  
- **이봉창_의거**: 김구는 이봉창을 한인애국단에 가입시켜 일본 천황 암살 거사를 준비시키고 수류탄과 자금을 마련해 도쿄로 파견했다.


`read_span`을 몇 번 불렀는지, 그래서 턴이 얼마나 늘었는지 본다.

In [38]:
spooled = show("오프로딩", answer, trace)

  턴 1  입력     142 토큰  ['read_doc', 'read_doc', 'read_doc']
  턴 2  입력     990 토큰  ['read_doc']
  턴 3  입력   1,425 토큰  ['read_span', 'read_span', 'read_span']
  턴 4  입력   7,481 토큰  ['read_span']
  턴 5  입력   8,612 토큰  []
오프로딩   턴 5  입력 합계  18,650 토큰   - **한인애국단**: 김구는 1931년 상하이에서 대한민국 임시정부 내 항일무장투쟁단체인 한인애국단을 조직하고 단장으로서 단원 모집과 거사를 지휘했다.  
- **윤봉길**: 김구는 윤봉길을 한인애국단에 가입시킨 뒤 1932년 상하이 홍커우 공원 의거를 계획하고 실행하도록 맡겼다.  
- **이봉창_의거**: 김구는 이봉창을 한인애국단에 가입시켜 일본 천황 암살 거사를 준비시키고 수류탄과 자금을 마련해 도쿄로 파견했다.


### 3. 필요할 때 가져온다

앞의 둘은 이미 넣은 것을 줄였다. 이건 애초에 문서 단위로 넣지 않는다.
1주차에서 만든 것이 이것이고, 여기서는 도구 하나로 들어온다.
인덱스는 이 노트북이 자기 안에서 만든다. 프레임워크 없이 임베딩과 내적뿐이다. (rag의 retriever와 유사하게 작동)

In [39]:
chunks, owner = [], []
for f in sorted(CORPUS.glob("*.md")):
    text = f.read_text(encoding="utf-8")
    for i in range(0, len(text), 600):
        chunks.append(text[i:i + 700])
        owner.append(f.stem)

print(f"청크 {len(chunks):,}개")

청크 1,352개


청크를 한꺼번에 임베딩하고 정규화해 둔다. 내적이 곧 코사인 유사도가 된다.

In [40]:
E = []
for i in range(0, len(chunks), 256):
    E += [d.embedding for d in keep(client.embeddings.create(model=EMBEDDING,
                                                             input=chunks[i:i + 256])).data]
E = np.array(E)
E = E / np.linalg.norm(E, axis=1, keepdims=True)
print(E.shape)

(1352, 1536)


검색 도구는 상위 다섯 청크를 돌려준다. 문서 전체는 컨텍스트에 들어가지 않는다.

In [41]:
def search(query):
    q = np.array(keep(client.embeddings.create(model=EMBEDDING, input=query)).data[0].embedding)
    q = q / np.linalg.norm(q)
    top = (E @ q).argsort()[::-1][:5]
    return json.dumps([{"title": owner[i], "text": chunks[i]} for i in top], ensure_ascii=False)


SEARCH = [spec("search", "질문과 가까운 본문 조각 다섯 개를 돌려준다.",
               {"query": {"type": "string", "description": "찾을 내용"}})]
print(json.loads(search("이봉창 의거에서 김구"))[0]["title"])

한인애국단


In [42]:
answer, trace = run(TASK, SEARCH, lambda n, a: search(**a))
print(answer)

- **한인애국단**: 김구는 1931년 대한민국 임시정부 산하에 한인애국단을 조직하고 단원을 모집해 항일 의거를 추진했다.  
- **윤봉길 의거**: 김구는 윤봉길을 한인애국단에 입단시킨 뒤 훙커우 공원 의거를 준비·지휘했다.  
- **이봉창 의거**: 김구는 이봉창을 한인애국단 1호 단원으로 가입시키고 일본 도쿄로 파견해 일왕 투척 의거를 실행하게 했다.


In [43]:
searched = show("검색", answer, trace)

  턴 1  입력      92 토큰  ['search', 'search', 'search']
  턴 2  입력   7,562 토큰  []
검색     턴 2  입력 합계   7,654 토큰   - **한인애국단**: 김구는 1931년 대한민국 임시정부 산하에 한인애국단을 조직하고 단원을 모집해 항일 의거를 추진했다.  
- **윤봉길 의거**: 김구는 윤봉길을 한인애국단에 입단시킨 뒤 훙커우 공원 의거를 준비·지휘했다.  
- **이봉창 의거**: 김구는 이봉창을 한인애국단 1호 단원으로 가입시키고 일본 도쿄로 파견해 일왕 투척 의거를 실행하게 했다.


### 비용 비교

여기서는 토큰만 측정한다. 답의 품질 비교에는 정답과 채점 기준이 필요하며 Part 4 `15`의 태스크셋에서 수행한다.

In [ ]:
for label, value in [("그대로", base), ("압축", packed), ("오프로딩", spooled), ("검색", searched)]:
    print(f"{label:6} {value:7,d} 토큰   그대로 대비 {value / base:5.1%}")

스풀 디렉터리를 정리한다.

In [ ]:
shutil.rmtree(SPOOL)

### 방법 선택 기준

| 방법 | 되돌릴 수 있나 | 무엇을 잃나 | 언제 |
|---|---|---|---|
| 압축 | 못 한다 | 요약이 버린 것 | 대화가 길어질 때. 오래된 턴부터 |
| 오프로딩 | 된다 | 없다. 대신 다시 가져오는 턴이 든다 | 결과가 크고 나중에 일부만 필요할 때 |
| 검색 | 된다 | 인덱스가 못 찾은 것 | 후보가 코퍼스만큼 많을 때 |

Session은 토큰 예산 안에 어떤 정보를 둘지 결정한다.

### 비용 확인

임베딩 usage에는 `output_tokens`가 없어 `total_tokens`로 계산한다.

In [ ]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p is None:
        continue
    if hasattr(u, "output_tokens"):
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]
    else:
        total += u.total_tokens / 1e6 * p[0]

print(f"호출 {len(USAGE)}회")
print(f"누적 ${total:.4f}")